In [149]:
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
import numpy as np

# SCRIPT_DIR = Path(__file__).absolute()

class MissingRequiredElementsError(Exception):
    """Raised when a file is syntactically valid but missing required elements"""
    pass

birdnet_results = pd.read_csv( Path(r"C:\Users\dereu\Documents\GitHub\DS4020-Birds\compiled\birdnet_master.csv"))
labelled = pd.read_csv(Path(r"C:\Users\dereu\Documents\GitHub\DS4020-Birds\data\labelled_bird_audio.dat"))



# Remove labelled files which are not represented in results.
filtered_labels = labelled.merge(
        birdnet_results, how="inner", 
        left_on=["Site", "Type", "Record_date", 'common_name'], 
        right_on=["location", "location_type", "recording_date", 'common_name'] )

# Check to make sure there are in fact labelled files represented in results. 
if len(filtered_labels) < 1:
    raise MissingRequiredElementsError("birdnet_master.csv does not contain data from any audio which has valid human-determined labels")

# Get aggrigated confidence metrics
reshaping = birdnet_results.pivot_table(    values='confidence',
                                            index = ["location", "location_type", "recording_date", "common_name"],
                                            aggfunc= ['sum', 'mean', 'max'],
                                            fill_value= 0
                                        )

reshaping2 = pd.DataFrame(reshaping)
reshaping2.columns = ['_'.join(col).strip() for col in reshaping.columns.to_flat_index()]
reshaping2

training = filtered_labels

# Presence indicator
training["detected"] = 1

# Collapse repeated detections within the same recording
grouped = (
    training.groupby(["location", "location_type", "recording_date", "common_name"], as_index=False)["detected"]
    .max()
)

species_list = pd.unique(grouped["common_name"]).tolist()

# Pivot to wide occupancy table
detections_table = grouped.pivot_table(
    index=["location", "location_type", "recording_date"],
    columns="common_name",
    values="detected",
    fill_value=0
).reset_index()

ungrouped = detections_table.melt(id_vars = ["location", "location_type", "recording_date"], value_vars = species_list)

combined_df = ungrouped.merge(
    reshaping2.reset_index(), how="left", 
)

#encoder = OneHotEncoder() -- As long as we use the same list of bird species for the detections table we 
#                               shouldn't have to worry about doing proper one-hot encoding
clf = RandomForestClassifier()

train_y = combined_df["value"]
#train_x = encoder.fit_transform(pd.DataFrame(combined_df["common_name"]))
#train_x = np.concatenate([np.array(combined_df), train_x])

train_x = pd.get_dummies(combined_df.fillna(0).loc[:,["common_name", "sum_confidence", "mean_confidence", "max_confidence"]])

clf.fit(train_x, train_y)

RandomForestClassifier()

In [ ]:
reshaping2 = pd.DataFrame(reshaping)
reshaping2.columns = ['_'.join(col).strip() for col in reshaping.columns.to_flat_index()]
reshaping2

In [93]:
ungrouped.merge(
    reshaping2.reset_index(), how="left", 
).to_csv("merged_df.csv")

In [91]:
birdnet_results

,SurveyID,Type,Record_date,common_name,genus,species,Minute_first_detected,Year,Month,Day,Site,Path,Filename
0,2,TER,2015-05-05,Horned Lark,Eremophila,alpestris,12.0,2015,May,5,ARM,ARM\ARM_TER_0413-09152015\STRIPS2_0+1_20150505...,STRIPS2_0+1_20150505_045100.flac
1,2,TER,2015-05-05,Sora,Porzana,carolina,12.0,2015,May,5,ARM,ARM\ARM_TER_0413-09152015\STRIPS2_0+1_20150505...,STRIPS2_0+1_20150505_045100.flac
2,2,TER,2015-05-05,Killdeer,Charadrius,vociferus,21.0,2015,May,5,ARM,ARM\ARM_TER_0413-09152015\STRIPS2_0+1_20150505...,STRIPS2_0+1_20150505_045100.flac
3,3,TER,2015-05-08,Horned Lark,Eremophila,alpestris,5.0,2015,May,8,ARM,ARM\ARM_TER_0413-09152015\STRIPS2_0+1_20150508...,STRIPS2_0+1_20150508_044700.flac
4,3,TER,2015-05-08,Unknown Bird,NaN,NaN,5.0,2015,May,8,ARM,ARM\ARM_TER_0413-09152015\STRIPS2_0+1_20150508...,STRIPS2_0+1_20150508_044700.flac
...,...,...,...,...,...,...,...,...,...,...,...,...,...
25299,1095,TER,2018-04-29,Common Grackle,Quiscalus,quiscula,58.0,2018,Apr,29,SLO,SLO\SLO_TER_0421-06192018\SLOA-TE_0+1_20180429...,SLOA-TE_0+1_20180429_055100.wav
25300,1095,TER,2018-04-29,Vesper Sparrow,Pooecetes,gramineus,58.0,2018,Apr,29,SLO,SLO\SLO_TER_0421-06192018\SLOA-TE_0+1_20180429...,SLOA-TE_0+1_20180429_055100.wav
25301,1095,TER,2018-04-29,American Robin,Turdus,migratorius,58.0,2018,Apr,29,SLO,SLO\SLO_TER_0421-06192018\SLOA-TE_0+1_20180429...,SLOA-TE_0+1_20180429_055100.wav
25302,1095,TER,2018-04-29,Horned Lark,Eremophila,alpestris,58.0,2018,Apr,29,SLO,SLO\SLO_TER_0421-06192018\SLOA-TE_0+1_20180429...,SLOA-TE_0+1_20180429_055100.wav


In [150]:
from sklearn.metrics import accuracy_score

predictions = clf.predict(train_x)

accuracy_score(predictions, train_y)

0.999972191323693

In [ ]:
print(np.array(combined_df).shape)
print(train_x.shape)

np.concatenate([combined_df, train_x], axis=0)



(71920, 8)
(71920, 80)


ValueError: all the input arrays must have same number of dimensions, but the array at index 0 has 2 dimension(s) and the array at index 1 has 0 dimension(s)

In [140]:
combined_df

,location,location_type,recording_date,common_name,value,sum_confidence,mean_confidence,max_confidence
0,ARM,CTL,2015-05-05,American Robin,1.0,33.727189,0.315207,0.825165
1,ARM,CTL,2015-05-12,American Robin,1.0,36.193646,0.254885,0.839113
2,ARM,CTL,2015-05-16,American Robin,1.0,4.257446,0.202736,0.544866
3,ARM,CTL,2015-05-25,American Robin,1.0,40.087588,0.263734,0.788131
4,ARM,CTL,2015-06-05,American Robin,1.0,32.603630,0.291104,0.897261
...,...,...,...,...,...,...,...,...
71915,WOR,EXP,2019-06-10,Downy Woodpecker,0.0,NaN,NaN,NaN
71916,WOR,EXP,2019-06-15,Downy Woodpecker,0.0,NaN,NaN,NaN
71917,WOR,EXP,2019-06-26,Downy Woodpecker,0.0,NaN,NaN,NaN
71918,WOR,EXP,2019-07-02,Downy Woodpecker,1.0,0.166863,0.166863,0.166863


In [148]:
pd.get_dummies(combined_df.fillna(0).loc[:,["common_name", "sum_confidence", "mean_confidence", "max_confidence"]])

,sum_confidence,mean_confidence,max_confidence,common_name_American Crow,common_name_American Goldfinch,common_name_American Robin,common_name_American Tree Sparrow,common_name_Baltimore Oriole,common_name_Barn Swallow,common_name_Barred Owl,...,common_name_Western Meadowlark,common_name_White-crowned Sparrow,common_name_White-throated Sparrow,common_name_Wild Turkey,common_name_Willow Flycatcher,common_name_Wilson's Snipe,common_name_Wood Duck,common_name_Yellow Warbler,common_name_Yellow-billed Cuckoo,common_name_Yellow-rumped Warbler
0,33.727189,0.315207,0.825165,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,36.193646,0.254885,0.839113,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,4.257446,0.202736,0.544866,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,40.087588,0.263734,0.788131,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,32.603630,0.291104,0.897261,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71915,0.000000,0.000000,0.000000,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
71916,0.000000,0.000000,0.000000,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
71917,0.000000,0.000000,0.000000,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
71918,0.166863,0.166863,0.166863,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
